In [10]:
import io
import zipfile
import requests
import frontmatter
from tqdm.auto import tqdm
import numpy as np
from sentence_transformers import SentenceTransformer
from minsearch import Index, VectorSearch

## 1. Load Repository Data

In [11]:
def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [12]:
moneyPrinterV2_docs = read_repo_data('FujiwaraChoki', 'MoneyPrinterV2')
print(f"MoneyPrinterV2 documents: {len(moneyPrinterV2_docs)}")

MoneyPrinterV2 documents: 11


## 2. Chunking with Sliding Window

In [13]:
def sliding_window(seq, size, step):
    """
    Split text into overlapping chunks.
    
    Args:
        seq: Input text string
        size: Size of each chunk in characters
        step: Step size between chunks (overlap = size - step)
    
    Returns:
        List of dicts with 'start' position and 'chunk' text
    """
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [14]:
moneyPrinter_chunks = []

for doc in moneyPrinterV2_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    moneyPrinter_chunks.extend(chunks)

print(f"Total chunks: {len(moneyPrinter_chunks)}")

Total chunks: 28


## 3. Text-Based Index (BM25)

In [15]:
text_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

text_index.fit(moneyPrinter_chunks)

## 4. Vector-Based Index (Semantic Search)

In [16]:
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [17]:
moneyPrinter_embeddings = []

for d in tqdm(moneyPrinter_chunks):
    v = embedding_model.encode(d['chunk'])
    moneyPrinter_embeddings.append(v)

moneyPrinter_embeddings = np.array(moneyPrinter_embeddings)

  0%|          | 0/28 [00:00<?, ?it/s]

In [18]:
vector_index = VectorSearch()
vector_index.fit(moneyPrinter_embeddings, moneyPrinter_chunks)

## 5. Hybrid Search Implementation

In [19]:
def text_search(query, num_results=5):
    """
    Perform BM25-based keyword search.
    """
    return text_index.search(query, num_results=num_results)

def vector_search(query, num_results=5):
    """
    Perform semantic vector search using embeddings.
    """
    query_embedding = embedding_model.encode(query)
    return vector_index.search(query_embedding, num_results=num_results)

def hybrid_search(query, num_results=5):
    """
    Combine text and vector search results.
    Returns deduplicated results from both approaches.
    """
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)
    
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        result_id = f"{result['filename']}_{result.get('start', 0)}"
        if result_id not in seen_ids:
            seen_ids.add(result_id)
            combined_results.append(result)
    
    return combined_results[:num_results * 2]

## 6. Test the Hybrid Search

In [20]:
query = "How to setup and use MoneyPrinterV2?"

print("=" * 80)
print(f"Query: {query}")
print("=" * 80)

results = hybrid_search(query, num_results=3)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}:")
    print(f"File: {result['filename']}")
    print(f"Title: {result.get('title', 'N/A')}")
    print(f"Start: {result.get('start', 0)}")
    print(f"Chunk preview: {result['chunk'][:200]}...")
    print("-" * 80)

Query: How to setup and use MoneyPrinterV2?

Result 1:
File: MoneyPrinterV2-main/CLAUDE.md
Title: N/A
Start: 0
Chunk preview: # CLAUDE.md

This file provides guidance to Claude Code (claude.ai/code) when working with code in this repository.

## Project Overview

MoneyPrinterV2 (MPV2) is a Python 3.12 CLI tool that automates...
--------------------------------------------------------------------------------

Result 2:
File: MoneyPrinterV2-main/docs/PostBridge.md
Title: N/A
Start: 0
Chunk preview: # Post Bridge Integration

MoneyPrinterV2 can optionally hand off a successfully uploaded YouTube Short to [Post Bridge](https://api.post-bridge.com/reference), which then publishes the same asset to ...
--------------------------------------------------------------------------------

Result 3:
File: MoneyPrinterV2-main/CODE_OF_CONDUCT.md
Title: N/A
Start: 1000
Chunk preview:  language
- Being respectful of differing viewpoints and experiences
- Gracefully accepting constructive criticism
- At

## 7. Compare Search Methods

In [21]:
def compare_search_methods(query):
    """
    Compare text, vector, and hybrid search results side by side.
    """
    print(f"Query: {query}")
    print("=" * 80)
    
    text_results = text_search(query, num_results=3)
    vector_results = vector_search(query, num_results=3)
    hybrid_results = hybrid_search(query, num_results=3)
    
    print(f"\n📝 TEXT SEARCH ({len(text_results)} results):")
    for i, r in enumerate(text_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")
    
    print(f"\n🔍 VECTOR SEARCH ({len(vector_results)} results):")
    for i, r in enumerate(vector_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")
    
    print(f"\n🔀 HYBRID SEARCH ({len(hybrid_results)} results):")
    for i, r in enumerate(hybrid_results, 1):
        print(f"{i}. {r['filename']} - {r.get('title', 'N/A')}")

In [22]:
compare_search_methods("How to install and configure the application?")

Query: How to install and configure the application?

📝 TEXT SEARCH (3 results):
1. MoneyPrinterV2-main/docs/Configuration.md - N/A
2. MoneyPrinterV2-main/README.md - N/A
3. MoneyPrinterV2-main/README.md - N/A

🔍 VECTOR SEARCH (3 results):
1. MoneyPrinterV2-main/docs/Configuration.md - N/A
2. MoneyPrinterV2-main/AGENTS.md - N/A
3. MoneyPrinterV2-main/AGENTS.md - N/A

🔀 HYBRID SEARCH (5 results):
1. MoneyPrinterV2-main/docs/Configuration.md - N/A
2. MoneyPrinterV2-main/README.md - N/A
3. MoneyPrinterV2-main/README.md - N/A
4. MoneyPrinterV2-main/AGENTS.md - N/A
5. MoneyPrinterV2-main/AGENTS.md - N/A


In [23]:
compare_search_methods("video generation features")

Query: video generation features

📝 TEXT SEARCH (3 results):
1. MoneyPrinterV2-main/docs/YouTube.md - N/A
2. MoneyPrinterV2-main/docs/Roadmap.md - N/A
3. MoneyPrinterV2-main/CLAUDE.md - N/A

🔍 VECTOR SEARCH (3 results):
1. MoneyPrinterV2-main/docs/YouTube.md - N/A
2. MoneyPrinterV2-main/docs/Roadmap.md - N/A
3. MoneyPrinterV2-main/docs/Configuration.md - N/A

🔀 HYBRID SEARCH (4 results):
1. MoneyPrinterV2-main/docs/YouTube.md - N/A
2. MoneyPrinterV2-main/docs/Roadmap.md - N/A
3. MoneyPrinterV2-main/CLAUDE.md - N/A
4. MoneyPrinterV2-main/docs/Configuration.md - N/A


In [39]:
import google.generativeai as genai
import os



from dotenv import load_dotenv
load_dotenv()

True

In [40]:
genai.configure(api_key=os.environ['GOOGLE_API_KEY'])

def llm_gemini(prompt, model='gemini-2.5-flash'):
    """
    Use Google's Gemini model for text generation.
    
    Models available:
    - gemini-2.5-flash: Fast, cheap, good for most tasks
    - gemini-2.5-pro: More capable, slower, more expensive
    - gemini-2.0-flash-exp: Latest experimental model
    """
    model_client = genai.GenerativeModel(model)
    response = model_client.generate_content(prompt)
    return response.text


In [57]:
user_prompt = "I want to print money"

llm_gemini(user_prompt)

ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 21.392147154s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerDayPerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.5-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 20
}
, retry_delay {
  seconds: 21
}
]

In [42]:
def text_search(query: str) -> list:
    """
    Search the FAQ using text/keyword matching.
    
    Args:
        query: Search query string
    
    Returns:
        List of top 5 matching results
    """
    return text_index.search(query, num_results=5)


In [48]:
text_search_tool = genai.protos.FunctionDeclaration(
    name="text_search",              
    description="Search the FAQ database",   
    parameters=genai.protos.Schema(  
        type=genai.protos.Type.OBJECT,   
        properties={
            "query": genai.protos.Schema(
                type=genai.protos.Type.STRING,
                description="Search query"
            )
        },
        required=["query"]
    )
)

In [53]:
system_prompt = """
You are a helpful assistant for a course. 
"""

# Create model with tool
model = genai.GenerativeModel(
    model_name='gemini-2.5-flash',
    system_instruction=system_prompt,
    tools=[text_search_tool]
)

In [54]:
chat = model.start_chat()
question = "I just discovered the course, can I join now?"
response = chat.send_message(question)
print(response)


response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "function_call": {
                  "name": "text_search",
                  "args": {
                    "query": "can I join the course now"
                  }
                }
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "index": 0
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 61,
        "candidates_token_count": 20,
        "total_token_count": 139
      },
      "model_version": "gemini-2.5-flash"
    }),
)


In [55]:
# Extract the function call from response
function_call = response.candidates[0].content.parts[0].function_call
# Get the query argument
query = function_call.args['query']
print(f"Gemini wants to search for: {query}")
# Execute YOUR search function
result = text_search(query=query)
print(f"Found {len(result)} results")


Gemini wants to search for: can I join the course now
Found 5 results


In [56]:
import json

# Package results for Gemini
function_response = genai.protos.Part(
    function_response=genai.protos.FunctionResponse(
        name="text_search",
        response={"result": json.dumps(result, default=str)}
    )
)

# Send back (chat remembers the full history automatically!)
final_response = chat.send_message(function_response)

# Print the final answer
print(final_response.text)

This course is actually a software project called MoneyPrinterV2. You can't "join" it in the traditional sense of enrolling in a course. Instead, you can set it up and use it.

To get started, you would need to:
1. Clone the GitHub repository: `git clone https://github.com/FujiwaraChoki/MoneyPrinterV2.git`
2. Navigate to the project directory: `cd MoneyPrinterV2`
3. Copy the example configuration and fill out your values in `config.json`: `cp config.example.json config.json`
4. Create and activate a virtual environment.
5. Install the required dependencies: `pip install -r requirements.txt`
6. Run the application: `python src/main.py`

You'll also need Python 3.12 and potentially the Go Programming Language if you plan to use certain features like reaching out to scraped businesses via email.

You can find more detailed documentation in the `docs/` directory of the project.


## 8. Pydantic AI Agent with Gemini (Day 4 - Agent)

Using Pydantic AI to simplify function calling. All the back-and-forth with Gemini is handled automatically.

In [ ]:
from typing import List, Any
from pydantic_ai import Agent
import time

from dotenv import load_dotenv
load_dotenv()

def text_search_tool(query: str) -> List[Any]:
    """
    Search the MoneyPrinterV2 documentation using keyword matching.

    Args:
        query: The search query string to look up in the docs.

    Returns:
        A list of up to 5 matching search results from the documentation.
    """
    return text_index.search(query, num_results=5)


system_prompt_v1 = """
You are a helpful assistant for the MoneyPrinterV2 project.

Use the search tool to find relevant information from the project documentation before answering questions.

If you can find specific information through search, use it to provide accurate answers.
If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

agent = Agent(
    name="mpv2_agent",
    system_prompt=system_prompt_v1,
    tools=[text_search_tool],
    model='google-gla:gemini-2.0-flash'
)

In [ ]:
question = "How do I install and configure MoneyPrinterV2?"
result = await agent.run(user_prompt=question)
print(result.output)

In [ ]:
result.new_messages()